In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
import chromadb

In [ ]:
pdf_path = "../rag_documents/Synthese_Politique_de_Retour_SHEIN_Maroc.pdf"

all_texts = []
all_metadatas = []
all_ids = []

loader = PyPDFLoader(pdf_path)
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "]
)
docs = text_splitter.split_documents(pages)

file_name = os.path.basename(pdf_path)

for i, doc in enumerate(docs):
    all_texts.append(doc.page_content)
    all_metadatas.append({
        "source": file_name,
        "page": doc.metadata.get("page", 0)
    })
    all_ids.append(f"rule_{i}")



8318


In [22]:
embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

chroma_client = chromadb.PersistentClient(path="./vector_db_rules")
collection = chroma_client.get_or_create_collection(name="business_rules_single_pdf")

vectors = embeddings_model.embed_documents(all_texts)

collection.add(
    documents=all_texts,
    embeddings=vectors,
    metadatas=all_metadatas,
    ids=all_ids
)

print(f"Embedding et indexation réussis ! {len(all_texts)} chunks ajoutés à ChromaDB.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11832.36it/s]


Embedding et indexation réussis ! 20 chunks ajoutés à ChromaDB.


In [ ]:
import os
from dotenv import load_dotenv
import psycopg2
from pgvector.psycopg2 import register_vector
import json

load_dotenv()

# Connexion à PostgreSQL
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
)

# 1. Enregistrer le vecteur pour la connexion
register_vector(conn)

# 2. Créer l'extension et la table
cur = conn.cursor()
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
conn.commit()

cur.execute("""
    CREATE TABLE IF NOT EXISTS business_rules_embeddings (
        id SERIAL PRIMARY KEY,
        chunk_id TEXT,
        content TEXT,
        metadata JSONB,
        embedding vector(384)
    );
""")
conn.commit()

# 3. Insérer les données
for text, vector, metadata, chunk_id in zip(all_texts, vectors, all_metadatas, all_ids):
    # Convertir le vecteur en liste s'il est au format numpy array
    if hasattr(vector, "tolist"):
        vector = vector.tolist()

    cur.execute(
        """
        INSERT INTO business_rules_embeddings (chunk_id, content, metadata, embedding)
        VALUES (%s, %s, %s, %s)
        """,
        (chunk_id, text, json.dumps(metadata), vector)
    )

conn.commit()
cur.close()
conn.close()

print("Stockage réussi dans PostgreSQL !")

Stockage réussi dans PostgreSQL !
